# AI Partner V1.3 - Custom AI + Persistent Memory

This notebook is the first V1.3 milestone. It keeps the V1.2 built-in character flow and adds guided custom AI creation, persistent memory, and custom dialogue examples.

Deferred for later milestones: RAG, LoRA/QLoRA training, adapter loading, export/import packages, and advanced evaluation.

## 1. Setup and Installation

Run this in Google Colab before importing the project modules.

In [ ]:
# Colab setup. Skip this cell if dependencies are already installed.
!pip install -q -r requirements.txt

## 2. Project Root Validation

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
required_paths = [
    Path('src'),
    Path('config'),
    Path('data'),
    Path('config/characters.json'),
    Path('config/config.json'),
    Path('requirements.txt'),
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required project paths: ' + ', '.join(missing))

for folder in [Path('data/custom_dialogues'), Path('memory')]:
    folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Project validation passed.')

## 3. Imports

In [ ]:
from pathlib import Path
import json

from src.character_select import load_characters
from src.custom_ai import (
    append_dialogue_examples,
    count_dialogue_pairs,
    create_custom_ai,
    ensure_custom_dialogue_file,
    save_custom_character,
)
from src.memory import ConversationMemory
from src.model import PersonaChatbot
from src.persona import Persona

CONFIG_PATH = Path('config/config.json')
CHARACTERS_PATH = Path('config/characters.json')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = json.load(f)

print('Imports ready.')

## 4. Choose Mode and Character

Choose an existing character, create a custom AI, or load an existing custom AI.

In [ ]:
def show_character_options(characters):
    print('Available characters:')
    for index, character in enumerate(characters, 1):
        label = 'custom' if character.get('is_custom') else character.get('dere_type', '')
        print(f"[{index}] {character['id']} - {character['display_name']} ({label})")

characters = load_characters(CHARACTERS_PATH)
print('[1] Use existing character')
print('[2] Create custom AI')
print('[3] Load custom AI')
mode = input('Choose mode [1-3]: ').strip() or '1'

if mode == '2':
    existing_ids = [character['id'] for character in characters]
    custom_character = create_custom_ai(existing_ids=existing_ids)
    character = save_custom_character(custom_character, CHARACTERS_PATH)
    dialogue_path = ensure_custom_dialogue_file(character)
    print(f"Created {character['display_name']} with dialogue file: {dialogue_path}")
elif mode == '3':
    custom_characters = [character for character in characters if character.get('is_custom')]
    if not custom_characters:
        raise ValueError('No custom characters found yet. Run mode 2 first.')
    show_character_options(custom_characters)
    selected = int(input(f'Choose custom character [1-{len(custom_characters)}]: ').strip()) - 1
    character = custom_characters[selected]
else:
    builtin_characters = [character for character in characters if not character.get('is_custom')]
    show_character_options(builtin_characters)
    selected = int(input(f'Choose built-in character [1-{len(builtin_characters)}]: ').strip() or '1') - 1
    character = builtin_characters[selected]

user_gender = input('Your gender for tone selection [male/female/neutral]: ').strip().lower() or 'neutral'
if user_gender not in {'male', 'female', 'neutral'}:
    user_gender = 'neutral'

character_id = character['id']
print('Selected:', character['display_name'])
print('Character id:', character_id)
print('User gender:', user_gender)

## 5. Persona Preview

In [ ]:
persona = Persona.from_config(CONFIG_PATH, character_id=character_id, user_gender=user_gender)

print('Name:', persona.name)
print('Gender:', persona.character_gender)
print('Age:', persona.age)
print('Type:', persona.dere_type)
print('Relationship mode:', persona.relationship_mode)
print('Language style:', persona.language_style)
print('Response length:', persona.response_length)
print('Greeting:', persona.greeting)
print('Dialogue file:', persona.dialogue_file)
print()
print('--- System prompt preview ---')
print(persona.build_prefix())

## 6. Memory Setup

In [ ]:
memory_cfg = config.get('memory', {})
memory_dir = Path(memory_cfg.get('memory_dir', 'memory'))
memory_dir.mkdir(parents=True, exist_ok=True)
memory_path = memory_dir / f'{character_id}_memory.json'

memory = ConversationMemory.from_config(CONFIG_PATH)
if memory_path.exists():
    load_existing = input(f'Load saved memory from {memory_path}? [y/N]: ').strip().lower()
    if load_existing == 'y':
        memory.load_memory(memory_path)
        print('Loaded memory turns:', memory.turn_count)
else:
    print('No saved memory yet for this character.')

print('Memory file:', memory_path)

## 7. Load Base Model

This is the expensive cell. It may take several minutes and works best with a Colab T4 GPU.

In [ ]:
chatbot = PersonaChatbot.from_config(
    CONFIG_PATH,
    persona=persona,
    memory=memory,
    character_id=character_id,
    user_gender=user_gender,
)
chatbot.save_memory_path = memory_path
print('Loaded model:', chatbot.model_name)

## 8. Optional Generation Settings

In [ ]:
# Tune generation without reloading the model. Leave values as-is or edit them.
chatbot.update_generation_settings(
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=120,
    repetition_penalty=1.1,
)
print('Generation settings updated.')

## 9. Chat

In [ ]:
print(persona.greeting)
print("Type 'exit', 'quit', or 'stop' to end the chat.")

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in {'exit', 'quit', 'stop'}:
        break
    if not user_input:
        continue
    reply = chatbot.generate(user_input, record=True)
    print(f'{persona.name}: {reply}')
    chatbot.memory.save_memory(memory_path, character_id=character_id)

print('Memory saved to:', memory_path)

## 10. Memory Controls

In [ ]:
print('Summary:')
print(memory.summary or '(empty)')
print()
print('Recent turns:')
for turn in memory.get_recent():
    print(f"{turn['role']}: {turn['content']}")

# Uncomment one of these actions when needed.
# memory.save_memory(memory_path, character_id=character_id)
# memory.load_memory(memory_path)
# memory.clear_memory(memory_path)

## 11. Add Custom Dialogue Examples

Use this for custom characters. Built-in character dialogue files are historical reference data, so avoid appending to them unless you intentionally want to edit those references.

In [ ]:
dialogue_path = Path(persona.dialogue_file)
if not character.get('is_custom'):
    print('Selected character is built-in. Create or load a custom AI before appending custom examples.')
else:
    print('Dialogue file:', dialogue_path)
    print('Current pair count:', count_dialogue_pairs(dialogue_path))
    examples_text = """USER: I had a difficult day.
CHARACTER: I am here with you. Tell me what happened, one piece at a time.
"""
    result = append_dialogue_examples(dialogue_path, examples_text)
    print('Added pairs:', result['added'])
    print('Total pairs:', result['total_pairs'])
    for warning in result['warnings']:
        print('Warning:', warning)

## 12. Basic V1.3 Validation

In [ ]:
characters = load_characters(CHARACTERS_PATH)
assert len([c for c in characters if not c.get('is_custom')]) >= 6
assert Path(persona.dialogue_file).exists() or not character.get('is_custom')
print('Built-in character count:', len([c for c in characters if not c.get('is_custom')]))
print('Custom character count:', len([c for c in characters if c.get('is_custom')]))
print('V1.3 milestone notebook validation passed.')

## Final Notes

This notebook intentionally does not include RAG, LoRA/QLoRA training, adapter loading, or export/import packaging. Those belong to later milestones after the custom AI and dataset workflow is stable.